## Streamlined methods to prepare weight categories for a HAF weighted analysis, updated to use the SDE design tracking schema

Author: Kaitlin Lubetkin, Alex Traynor
Created: 8/25/23  
Last Edited: 09/06/24  

## 1) Prep workspace and define functions 
<code style="background:yellow;color:black">***(run straight through, no edits needed)***</code>

In [ ]:
import os
import arcpy
import datetime

map = arcpy.mp.ArcGISProject("CURRENT").listMaps("Map")[0]
albers = 'PROJCS["NAD_1983_Albers",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",29.5],PARAMETER["Standard_Parallel_2",45.5],PARAMETER["Latitude_Of_Origin",23.0],UNIT["Meter",1.0]]'

# Function to split out design polygons by season
def split_strata(useAIM, useLMF):
    # Create analysis gdb
    arcpy.management.CreateFileGDB(os.path.join(parentFolder, ("Analysis_"+ analysis_name + ".gdb"))
    analysisGDB = os.path.join(parentFolder, ("Analysis_" + analysis_name + ".gdb"))
    
    # If AIM points are being used, clip to SUA and split out design polygons
    if useAIM:
        # TODO: subset to strata with sampled AIM points?
        arcpy.analysis.Clip(sddsde + "\\DesignPolygons",
                            aoi, 
                            "memory\\tempClip")
        arcpy.analysis.SplitByAttributes("tempClip", analysisGDB, "DesignName")
        arcpy.management.Delete("memory\\tempClip")
    
    # If LMF points are being used, clip to SUA and add
    if useLMF:
        arcpy.analysis.Clip(sddsde + "\\LMFStrata", 
                            aoi, 
                            analysisGDB + "\\" + "LMF")
        arcpy.management.AlterField(analysisGDB + "\\" + "LMF", "Stratum", "StratumLMF")
        map.removeLayer(map.listLayers()[0])
        
    print("All designs split out")
    
# Function to union design polygons
def union_strata():
    analysisGDB = os.path.join(parentFolder, ("Analysis_" + analysis_name + ".gdb"))
    arcpy.env.workspace = analysisGDB
    
    # Prepare union of all polygons
    designList = [os.path.join(analysisGDB, fc) for fc in arcpy.ListFeatureClasses()]
    polygonUnion = "tempUNION"
    arcpy.analysis.Union(designList, polygonUnion, "NO_FID")
    
    # Repair geometry
    arcpy.management.RepairGeometry(polygonUnion, "DELETE_NULL")
    print("    strata unioned and geometry repaired")
    
    # Add and calculate a field for the area in hectares
    arcpy.management.AddField(polygonUnion, "area_hectares", "DOUBLE")
    arcpy.management.CalculateGeometryAttributes(polygonUnion, 
                                                 "area_hectares AREA_GEODESIC", 
                                                 area_unit = "HECTARES")
    
    # Use Eliminate as first pass to clean up slivers < 1 ha
    #first, check if slivers are present
    arcpy.management.SelectLayerByAttribute("tempUNION", "CLEAR_SELECTION")
    tot = int(arcpy.management.GetCount("tempUNION")[0])
    arcpy.management.SelectLayerByAttribute("tempUNION", "NEW_SELECTION", "area_hectares < 1")
    sliv = int(arcpy.management.GetCount("tempUNION")[0])
    eliminated = "tempUNION"
    if sliv > 0 and sliv < tot:
        arcpy.management.Eliminate("tempUNION", os.path.join("memory", "tempElim"), "LENGTH")
        arcpy.management.Delete(os.path.join(analysisGDB, polygonUnion))
        eliminated = "tempElim"
        print("    Eliminate run to remove slivers")
        
        #if slivers are still present, run Eliminate a second time
        tot = int(arcpy.management.GetCount("tempElim")[0])
        arcpy.management.SelectLayerByAttribute("tempElim", "NEW_SELECTION", "area_hectares < 1")
        sliv = int(arcpy.management.GetCount("tempElim")[0])
        if sliv > 0 and sliv < tot:
            arcpy.management.Eliminate("tempElim", os.path.join("memory", "tempElim2"), "LENGTH")
            arcpy.management.Delete(os.path.join("memory", "tempElim"))
            eliminated = "tempElim2"
            print("    Eliminate run a second time to remove remaining slivers")
    else:
        print("    No slivers < 1 ha present")

    # Recalculate hectares
    arcpy.management.SelectLayerByAttribute(eliminated, "CLEAR_SELECTION")
    arcpy.management.CalculateGeometryAttributes(eliminated, 
                                                 "area_hectares AREA_GEODESIC", 
                                                 area_unit = "HECTARES")
    # Get all Polygon field names
    fields = arcpy.ListFields(eliminated)
    polygonIDFields = []
    for f in fields:
        if f.name=="StratumLMF":
            polygonIDFields.insert(0, f.name)
        elif f.name.startswith("DesignPolygonID"):
            polygonIDFields.append(f.name)
    
    # Add and calculate a field for the combined strata
    arcpy.management.AddField(eliminated, "strata_combo", "TEXT", field_length = 500)
    addr = ["!" + idField + "!" for idField in polygonIDFields] 
    arcpy.management.CalculateField(eliminated, "strata_combo", 
                                    "ConcatAddr(" + ','.join([a for a in addr]) + ")", 
                                    "PYTHON3", 
                                    "def ConcatAddr(*args): return ''.join([str(i) for i in args if i not in(None,' ')]) ", 
                                    "TEXT", 
                                    "NO_ENFORCE_DOMAINS")
     
    # Save new, cleaned up copy
    fm = ('Shape_Length "Shape_Length" false true true 8 Double 0 0,First,#,' + eliminated + ',Shape_Length,-1,-1;' + 
          'Shape_Area "Shape_Area" false true true 8 Double 0 0,First,#,' + eliminated + ',Shape_Area,-1,-1;' + 
          'area_hectares_' + ' "area_hectares_' + '" true true false 8 Double 0 0,First,#,' + eliminated + ',area_hectares,-1,-1;' + 
          'strata_combo "strata_combo" true true false 255 Text 0 0,First,#,' + eliminated + ',strata_combo,0,500;')
    arcpy.conversion.FeatureClassToFeatureClass(eliminated, analysisGDB, "wgtCats", field_mapping=fm)
    arcpy.management.Project(os.path.join(analysisGDB, "wgtCats"), 
                             os.path.join(analysisGDB, "designPolygons_UNION_" + "_WgtCats"), 
                             albers)
    arcpy.management.Delete(eliminated)    
    arcpy.management.Delete(os.path.join(analysisGDB, "wgtCats"))
    if arcpy.Exists("memory\\tempElim"): arcpy.management.Delete("memory\\tempElim")
    if arcpy.Exists("memory\\tempElim2"): arcpy.management.Delete("memory\\tempElim2")
    print("    cleaned union created")

# Function to finalize UNION eliminated version and calculate weight categories
def finalize_wgtCats():
    #Calculate hectares
    arcpy.management.CalculateGeometryAttributes("designPolygons_UNION_" +  "_WgtCats", 
                                                 "area_hectares_" + " AREA_GEODESIC", 
                                                 area_unit = "HECTARES")
    #Add weight cat field
    arcpy.management.AddField("designPolygons_UNION_" + "_WgtCats", "wgtcat_", "SHORT")
    with arcpy.da.UpdateCursor("designPolygons_UNION_" +  "_WgtCats", "wgtcat_") as cursor:
        i = 1
        for row in cursor:
            row[0] = i
            i += 1
            cursor.updateRow(row)
        del row
    del cursor

## 2) Define variables
<code style="background:yellow;color:black">***(edit for specific analysis)***</code>

In [ ]:
# Parent/root folder for analysis
parentFolder = (r'C:\Users\alaurencetraynor\Documents\Tools')
# Analysis name
analysis_name = "example"
# Analysis AOI
aoi = "Benchmark Groups"
# SDD geodatabase - this should be a copy you'll use for your analysis
sddsde = os.path.join(parentFolder, ("SDDSDE_"+ analysis_name +".gdb"))

# project everything into Albers
for fc in arcpy.ListFeatureClasses(feature_type="Polygon"):
    arcpy.management.Project(os.path.join(tool1GDB, fc), os.path.join(analysisGDB, fc), albers)

## 3) Prepare union of design polygons
<code style="background:yellow;color:black">***(edit for specific analysis)***</code>

In [ ]:
# Set which designs are applicable for each season
useAIM = True
useLMF = True

# Union the dang thang
split_strata(useAIM, useLM)
union_strata()
print("Done!")

In [ ]:
## Next, check remaining slivers and very small not-quite-slivers (1-1.5 ha) and merge
# with adjacent larger polygons as appropriate

# Then proceed with next Jupyter cell

In [ ]:
## Finalize each weight category
finalize_wgtCats()

In [ ]:
## Double check small (1-2 ha) polygons
## If necessary, return to the UNION and merge slivers with adjacent larger polygons, then redo the finalize_wgtCats

## 4) Prepare weight category polygons for analysis
<code style="background:yellow;color:black">***(run straight through, no edits needed)***</code>

In [ ]:
## Grab sampled points from SDE/benchmark tool output and reproject into Albers
# if you havent run through benchmark took, this will just be the terrestrial points lyer from the SDE
points = r"\\blm.doi.net\dfs\loc\EGIS\ProjectsNational\AIM\AIMDataTools\SDE\AIMTerrestrialPub.sde"

arcpy.management.SelectLayerByLocation(in_layer = points,
                                        overlap_type="INTERSECT", 
                                        select_features= aoi, 
                                        selection_type="NEW_SELECTION")
    
    arcpy.management.DeleteFeatures(fineScale + thisSeason[0])
    # Add FinalDesignation = Target Sampled for all sampled points
    arcpy.management.AddField(fineScale + thisSeason[0], "FinalDesignation", "TEXT", 255)
    arcpy.management.CalculateField(fineScale + thisSeason[0], "FinalDesignation", '"Target Sampled"', "PYTHON3")
    
    # Add NotSampled points
    NotSampled_temp = arcpy.management.MakeFeatureLayer(in_features=sddsde + "\\NotSampled", out_layer="memory\\NotSampled_temp")
    arcpy.management.SelectLayerByLocation(in_layer=NotSampled_temp,
                                           overlap_type="INTERSECT", 
                                           select_features= "GRSG_Boundaries\\" +  fineScale + thisSeason[1] + "_SUA", 
                                           selection_type="NEW_SELECTION")
    fm = ('PlotID "PlotID" true true false 255 Text 0 0,First,#,NotSampled_temp,PlotID,0,255;' + 
          'PlotKey "PlotKey" true true false 255 Text 0 0,First,#,NotSampled_temp,PlotKey,0,255;' + 
          'PrimaryKey "PrimaryKey" true true false 255 Text 0 0,First,#,NotSampled_temp,DesignPointKey,0,255;' + 
          'DateVisited "DateVisited" true true false 8 Date 0 0,First,#,NotSampled_temp,DateEvaluated,-1,-1;' + 
          'State "State" true true false 2 Text 0 0,First,#,NotSampled_temp,ADMIN_ST,0,2;' + 
          'FinalDesignation "FinalDesignation" true true false 255 Text 0 0,First,#,NotSampled_temp,FinalDesignation,0,255;')
    arcpy.management.Append(NotSampled_temp, fineScale + thisSeason[0], "NO_TEST", fm)
    map.removeLayer(map.listLayers()[0])    
    
    # Add fields for weight cat, segment, and area_ratio
    arcpy.management.AddField(fineScale + thisSeason[0], "wgtcat" + thisSeason[1], "SHORT")
    arcpy.management.AddField(fineScale + thisSeason[0], "segment_id", "TEXT", 20)
    arcpy.management.AddField(fineScale + thisSeason[0], "area_ratio" + thisSeason[1], "DOUBLE")
    
    # Spatially join to seasonal WgtCats and calculate weight cat
    arcpy.analysis.SpatialJoin(fineScale + thisSeason[0], 
                               "designPolygons_UNION" + thisSeason[1] + "_WgtCats", 
                               os.path.join(analysisGDB, "spjoin" + thisSeason[1]))
    arcpy.management.AddJoin(fineScale + thisSeason[0], "OBJECTID", 
                             "spjoin" + thisSeason[1], "TARGET_FID", "KEEP_ALL")
    arcpy.management.CalculateField(fineScale + thisSeason[0], 
                                    "wgtcat" + thisSeason[1], 
                                    "!spjoin" + thisSeason[1] + ".wgtcat" + thisSeason[1] + "_1!", "PYTHON3")
    arcpy.management.RemoveJoin(fineScale + thisSeason[0], "spjoin" + thisSeason[1])
    arcpy.management.Delete(os.path.join(analysisGDB, "spjoin" + thisSeason[1]))
    
    # Spatially join to seasonal segments and calculate segment_id & area_ratio
    arcpy.analysis.SpatialJoin(fineScale + thisSeason[0], 
                               sddsde + "\\LMFSegmentPolygons" + thisSeason[1], 
                               analysisGDB + "\\LMFseg_spjoin" + thisSeason[1])
    arcpy.management.AddJoin(fineScale + thisSeason[0], "OBJECTID", 
                             "LMFseg_spjoin" + thisSeason[1], "TARGET_FID", "KEEP_ALL")
    arcpy.management.CalculateField(fineScale + thisSeason[0], "segment_id",
                                    "!LMFseg_spjoin" + thisSeason[1] + ".SegmentPolygonID!", "PYTHON3")
    arcpy.management.CalculateField(fineScale + thisSeason[0], "area_ratio" + thisSeason[1], 
                                    "!LMFseg_spjoin" + thisSeason[1] + ".area_ratio!", "PYTHON3")
    arcpy.management.RemoveJoin(fineScale + thisSeason[0], "LMFseg_spjoin" + thisSeason[1])
    arcpy.management.Delete(analysisGDB + "\\LMFseg_spjoin" + thisSeason[1])
   
print("Done!")

## Finalize: 
&emsp;**Double check that none of the seasonal points have NULL for wgt cat**  
&emsp;**Double check that none of the seasonal LMF points have NULL for area_ratio or segment_id** 
### Then, go to the seasonal rmd scripts.